# 01 — Data Preparation & Noise Injection
## Self-Healing Neural Network | CIFAR-100

This notebook prepares CIFAR-100 on CPU, explores class statistics, and demonstrates on-the-fly image corruption used by the self-healing pipeline.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
import yaml
from torchvision import transforms

sys.path.append(str(Path('..').resolve()))
np.random.seed(42)
torch.manual_seed(42)

print('Working directory:', os.getcwd())
print('PyTorch version:', torch.__version__)

In [ ]:
config_path = Path('../configs/config.yaml')
with open(config_path, 'r', encoding='utf-8') as file:
    config = yaml.safe_load(file)

print('Loaded config:', config_path.resolve())
print('Dataset config:', config['dataset'])
print('Noise config:', config['noise'])

In [ ]:
base_transform = transforms.ToTensor()
train_raw = torchvision.datasets.CIFAR100(
    root=config['dataset']['root'], train=True, download=True, transform=base_transform
)
test_raw = torchvision.datasets.CIFAR100(
    root=config['dataset']['root'], train=False, download=True, transform=base_transform
)

sample_image, sample_label = train_raw[0]
print(f'Train size: {len(train_raw):,}')
print(f'Test size: {len(test_raw):,}')
print('Image shape:', tuple(sample_image.shape))
print('Num classes:', len(train_raw.classes))

In [ ]:
targets = np.array(train_raw.targets)
counts = np.bincount(targets, minlength=100)

fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(np.arange(100), counts, color='#2A9D8F')
ax.set_title('CIFAR-100 Class Distribution (Train Set)')
ax.set_xlabel('Class Index')
ax.set_ylabel('Count')
ax.grid(alpha=0.2)
plt.show()

print('All 100 class names:')
for idx, class_name in enumerate(train_raw.classes):
    end_char = '\n' if (idx + 1) % 10 == 0 else '\t'
    print(f'{idx:02d}:{class_name}', end=end_char)
print()

In [ ]:
# CIFAR-100 has 20 superclasses; here we show 30 diverse classes in a 3x10 grid.
selected_indices = []
seen_labels = set()
for idx, label in enumerate(train_raw.targets):
    if label not in seen_labels:
        seen_labels.add(label)
        selected_indices.append(idx)
    if len(selected_indices) == 30:
        break

fig, axes = plt.subplots(3, 10, figsize=(20, 6))
for ax, idx in zip(axes.flatten(), selected_indices):
    image, label = train_raw[idx]
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title(train_raw.classes[label], fontsize=8)
    ax.axis('off')

plt.suptitle('Clean CIFAR-100 Samples (3x10)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from src.dataset import NoiseInjector

injector = NoiseInjector()
image, label = train_raw[42]
gaussian = injector.gaussian_noise(image, std=config['noise']['gaussian_std'])
salt_pepper = injector.salt_pepper(image, prob=config['noise']['salt_pepper_prob'])
occlusion = injector.occlusion(image, patch_size=config['noise']['occlusion_size'])

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
titles = ['Original', 'Gaussian', 'Salt & Pepper', 'Occlusion']
images = [image, gaussian, salt_pepper, occlusion]

for ax, title, img in zip(axes, titles, images):
    ax.imshow(img.permute(1, 2, 0).clamp(0, 1))
    ax.set_title(title)
    ax.axis('off')

plt.suptitle(f'Noise Injection Demo | Class: {train_raw.classes[label]}')
plt.tight_layout()
plt.show()

In [ ]:
noise_levels = [0.1, 0.2, 0.3, 0.5, 0.7]
image, label = train_raw[42]

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
axes[0].imshow(image.permute(1, 2, 0).clamp(0, 1))
axes[0].set_title('Original')
axes[0].axis('off')

for i, std in enumerate(noise_levels, start=1):
    noisy = injector.gaussian_noise(image, std=std)
    axes[i].imshow(noisy.permute(1, 2, 0).clamp(0, 1))
    axes[i].set_title(f'std={std}')
    axes[i].axis('off')

plt.suptitle('Gaussian Noise Severity Comparison')
plt.tight_layout()
plt.show()

In [ ]:
from src.dataset import get_dataloaders

train_loader, val_loader, test_loader = get_dataloaders(config)
noisy_batch, clean_batch, labels_batch = next(iter(train_loader))

print('Noisy batch shape :', tuple(noisy_batch.shape))
print('Clean batch shape :', tuple(clean_batch.shape))
print('Labels batch shape:', tuple(labels_batch.shape))
print('Val batches:', len(val_loader), '| Test batches:', len(test_loader))

## Summary

| Metric | Value |
|---|---|
| Dataset | CIFAR-100 |
| Number of classes | 100 |
| Image size | 32 x 32 RGB |
| Train set size | 50,000 |
| Test set size | 10,000 |
| Noise types | Gaussian, Salt & Pepper, Occlusion |
| Data pipeline output | (noisy_image, clean_image, label) |